# Dự báo Doanh số Thương mại Điện tử
## Giai đoạn 1 — Nền tảng Đánh giá & Chuẩn hóa
---

## Import thư viện

In [1]:
import numpy as np
import pandas as pd

---
## Giai đoạn 1A — Evaluation Metrics
Cung cấp thước đo để đánh giá độ lệch giữa doanh số dự báo `ŷ` và doanh số thực tế `y`.

In [2]:
def calculate_mse(y_true, y_pred):
    """Mean Squared Error — dùng làm Loss Function"""
    n = len(y_true)
    return (1 / n) * np.sum((y_true - y_pred) ** 2)

def calculate_rmse(y_true, y_pred):
    """Root Mean Squared Error — đưa sai số về cùng đơn vị gốc"""
    return np.sqrt(calculate_mse(y_true, y_pred))

def calculate_mae(y_true, y_pred):
    """Mean Absolute Error — ít nhạy cảm với outlier hơn MSE"""
    n = len(y_true)
    return (1 / n) * np.sum(np.abs(y_true - y_pred))

def calculate_r2(y_true, y_pred):
    """R-squared — càng gần 1 càng tốt, âm = tệ hơn dự báo bằng mean"""
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    return 1 - (ss_res / ss_tot)

print("Đã định nghĩa xong các hàm Evaluation Metrics")

Đã định nghĩa xong các hàm Evaluation Metrics


In [3]:
# Kiểm tra nhanh Evaluation Metrics
y_true = np.array([100, 200, 300, 400, 500], dtype=float)
y_pred = np.array([110, 190, 310, 390, 510], dtype=float)

print("── Kiểm tra Evaluation Metrics ──")
print(f"MSE  : {calculate_mse(y_true, y_pred):.2f}   (kỳ vọng: 100.00)")
print(f"RMSE : {calculate_rmse(y_true, y_pred):.2f}   (kỳ vọng: 10.00)")
print(f"MAE  : {calculate_mae(y_true, y_pred):.2f}   (kỳ vọng: 10.00)")
print(f"R²   : {calculate_r2(y_true, y_pred):.4f}  (kỳ vọng: ~0.9975)")

── Kiểm tra Evaluation Metrics ──
MSE  : 100.00   (kỳ vọng: 100.00)
RMSE : 10.00   (kỳ vọng: 10.00)
MAE  : 10.00   (kỳ vọng: 10.00)
R²   : 0.9950  (kỳ vọng: ~0.9975)


---
## Giai đoạn 1B — CustomStandardScaler
Chuẩn hóa các đặc trưng về cùng hệ quy chiếu bằng Z-score: `Z = (X - µ) / σ`

> **Quan trọng:** Chỉ `fit()` trên tập **Train**. Tập Test chỉ được `transform()` bằng µ/σ đã học từ Train — tránh **data leakage**.

In [4]:
class CustomStandardScaler:
    def __init__(self):
        self.mean_ = None  # µ — lưu lại sau fit()
        self.std_  = None  # σ — lưu lại sau fit()

    def fit(self, X):
        """Tính µ và σ từ tập Train"""
        self.mean_ = np.mean(X, axis=0)
        self.std_  = np.std(X, axis=0)
        # Tránh chia cho 0 nếu feature có std = 0
        self.std_[self.std_ == 0] = 1
        return self

    def transform(self, X):
        """Áp dụng Z-score: Z = (X - µ) / σ"""
        if self.mean_ is None:
            raise RuntimeError("Cần gọi fit() trước khi transform()")
        return (X - self.mean_) / self.std_

    def fit_transform(self, X):
        """Gọi fit() rồi transform() — tiện cho tập Train"""
        return self.fit(X).transform(X)

print("Đã định nghĩa xong CustomStandardScaler")

Đã định nghĩa xong CustomStandardScaler


In [5]:
# Kiểm tra nhanh CustomStandardScaler
X_sample = np.array([[1000, 5], [2000, 10], [3000, 15], [4000, 20]], dtype=float)

scaler_test = CustomStandardScaler()
X_scaled = scaler_test.fit_transform(X_sample)

print("── Kiểm tra CustomStandardScaler ──")
print(f"Mean sau chuẩn hóa (kỳ vọng ≈ 0): {X_scaled.mean(axis=0).round(6)}")
print(f"Std  sau chuẩn hóa (kỳ vọng ≈ 1): {X_scaled.std(axis=0).round(6)}")

── Kiểm tra CustomStandardScaler ──
Mean sau chuẩn hóa (kỳ vọng ≈ 0): [0. 0.]
Std  sau chuẩn hóa (kỳ vọng ≈ 1): [1. 1.]


---
## Áp dụng lên dữ liệu thực tế

In [ ]:
# Load dữ liệu — thay đường dẫn cho đúng với máy của bạn
X_train = pd.read_csv("dataset_ready/X_train.csv").values.astype(float)
X_test  = pd.read_csv("dataset_ready/X_test.csv").values.astype(float)

print(f"X_train : {X_train.shape}")
print(f"X_test  : {X_test.shape}")

# Kiểm tra NaN
assert not np.isnan(X_train).any(), "X_train có giá trị NaN!"
assert not np.isnan(X_test).any(),  "X_test có giá trị NaN!"
print("\nDữ liệu hợp lệ, không có NaN")

FileNotFoundError: [Errno 2] No such file or directory: 'data/X_train.csv'

In [ ]:
# Chuẩn hóa
scaler     = CustomStandardScaler()
X_train_sc = scaler.fit_transform(X_train)  # fit chỉ trên train
X_test_sc  = scaler.transform(X_test)       # transform test bằng µ/σ của train

print("── Thông số Scaler ──")
print(f"µ (mean): {scaler.mean_}")
print(f"σ (std) : {scaler.std_}")
print("\nChuẩn hóa hoàn tất — X_train_sc và X_test_sc sẵn sàng cho Giai đoạn 2")